# Our PR2 MJCF in the Multiverse apartment

This notebook uses `MJCFParser` for both repository-local inputs:

- `semantic_digital_twin/resources/mujoco_resources/pr2/pr2.xml`
- `semantic_digital_twin/resources/mujoco_resources/apartment/apartment.xml`

It launches PR2 with parked arms, a low torso, and closed kitchen drawers. Milk and cereal are staged on the right countertop. The final cells define and optionally execute a simple PyCRAM plan that raises the torso and moves the robot forward.

In [ ]:
import os
import time
from dataclasses import dataclass
from pathlib import Path
from xml.etree import ElementTree as ET

import mujoco
import numpy as np

from physics_simulators.mujoco_simulator import MujocoSimulator
from pycram.datastructures.dataclasses import Context
from pycram.plans.factories import sequential
from pycram.robot_plans.actions.base import ActionDescription
from semantic_digital_twin.adapters.mjcf import MJCFParser
from semantic_digital_twin.adapters.multi_sim import MujocoBuilder, MujocoSynchronizer
from semantic_digital_twin.datastructures.definitions import StaticJointState, TorsoState
from semantic_digital_twin.datastructures.prefixed_name import PrefixedName
from semantic_digital_twin.robots.pr2 import PR2
from semantic_digital_twin.semantic_annotations.semantic_annotations import Cereal, Milk
from semantic_digital_twin.spatial_types import HomogeneousTransformationMatrix
from semantic_digital_twin.world_description.connections import ActiveConnection1DOF, FixedConnection
from semantic_digital_twin.world_description.geometry import Color, Scale

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'semantic_digital_twin/resources/mujoco_resources').is_dir():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the repository.')

REPO_ROOT = find_repo_root()
MUJOCO_RESOURCES = REPO_ROOT / 'semantic_digital_twin/resources/mujoco_resources'
PR2_MJCF = (MUJOCO_RESOURCES / 'pr2/pr2.xml').resolve()
APARTMENT_MJCF = (MUJOCO_RESOURCES / 'apartment/apartment.xml').resolve()
GENERATED_SCENE = Path('/tmp/pr2_multiverse_apartment.xml')
HEADLESS = os.environ.get('MUJOCO_HEADLESS', '0') == '1'

if not APARTMENT_MJCF.is_file():
    raise FileNotFoundError(APARTMENT_MJCF)
print(f'PR2:       {PR2_MJCF}')
print(f'Apartment: {APARTMENT_MJCF}')
print(f'Output:    {GENERATED_SCENE}')
print(f'Headless:  {HEADLESS}')

## Parse and merge both MJCF worlds

In [ ]:
pr2_world = MJCFParser(str(PR2_MJCF)).parse()

apartment_parser = MJCFParser(str(APARTMENT_MJCF))
# Both parsers otherwise create a root body named 'world'.
apartment_parser.spec.worldbody.name = 'apartment_world'
apartment_world = apartment_parser.parse()

pr2_world.merge_world(
    apartment_world,
    root_connection=FixedConnection(
        parent=pr2_world.root, child=apartment_world.root
    ),
)
world = pr2_world
print(f'{len(world.bodies)} bodies')
print(f'{len(world.connections)} connections')
print(f'{len(world.actuators)} actuators')

## Add PR2 semantics and counter objects

The generated MJCF omits decorative SRDF-only links, so the PR2 semantic annotations are initialized without loading the collision-ignore SRDF. Milk and cereal are fixed primitive boxes for now; later pick/place work can replace their fixed connections with free joints.

In [ ]:
with world.modify_world():
    pr2 = PR2._init_empty_robot(world)
    pr2._setup_semantic_annotations()
    pr2._setup_velocity_limits()
    pr2._setup_hardware_interfaces()
    pr2._setup_joint_states()
    world.add_semantic_annotation(pr2)

    milk = Milk.create_with_new_body_in_world(
        name=PrefixedName('milk'),
        world=world,
        scale=Scale(0.08, 0.08, 0.22),
        world_root_T_self=HomogeneousTransformationMatrix.from_xyz_rpy(
            3.0, 1.70, 1.105, reference_frame=world.root
        ),
    )
    cereal = Cereal.create_with_new_body_in_world(
        name=PrefixedName('cereal'),
        world=world,
        scale=Scale(0.16, 0.06, 0.28),
        world_root_T_self=HomogeneousTransformationMatrix.from_xyz_rpy(
            3.0, 2.05, 1.125, reference_frame=world.root
        ),
    )

milk.root.visual.dye_shapes(Color(0.85, 0.95, 1.0, 1.0))
cereal.root.visual.dye_shapes(Color(0.95, 0.65, 0.15, 1.0))
context = Context(world=world, robot=pr2)
context.evaluate_conditions = False
print(f'Objects: {milk.root.name.name}, {cereal.root.name.name}')

## Initial PR2 and apartment state

The planar base starts in the clear kitchen aisle. Both arms use the PR2 semantic parked state, the torso starts low, and every active drawer or door joint starts at its zero-valued closed position.

In [ ]:
# Clear aisle between the wall cabinets and the kitchen island.
PR2_X = 1.2
PR2_Y = 3.7
PR2_YAW = -1.5707963267948966

initial_joint_targets = {
    'base_x_joint': PR2_X,
    'base_y_joint': PR2_Y,
    'base_yaw_joint': PR2_YAW,
}
for arm in pr2.arms:
    parked = arm.get_joint_state_by_type(StaticJointState.PARK)
    initial_joint_targets.update(
        {connection.name.name: value for connection, value in parked.items()}
    )
initial_joint_targets.update(
    {
        connection.name.name: value
        for connection, value in pr2.torso.get_joint_state_by_type(TorsoState.LOW).items()
    }
)

# The source kitchen has zero as the closed position for every drawer and door.
closed_apartment_joints = [
    connection.name.name
    for connection in world.connections
    if isinstance(connection, ActiveConnection1DOF)
    and ('drawer' in connection.name.name or 'door' in connection.name.name)
]
initial_joint_targets.update({name: 0.0 for name in closed_apartment_joints})
for name, value in initial_joint_targets.items():
    world.get_connection_by_name(name).position = value
world.notify_state_change()
print(f'Initialized {len(closed_apartment_joints)} drawer/door joints as closed.')

## Build one MJCF scene and add lighting

The builder subclass enables the same inertia balancing used by our PR2 MJCF. The XML post-processing adds the headlight, skybox, two directional lights, gravity, and an implicit integrator.

In [ ]:
class BalancedMujocoBuilder(MujocoBuilder):
    def _start_build(self, file_path):
        super()._start_build(file_path)
        self.spec.compiler.balanceinertia = True
        self.spec.compiler.boundmass = 1e-6
        self.spec.compiler.boundinertia = 1e-6

BalancedMujocoBuilder().build_world(world, str(GENERATED_SCENE))

tree = ET.parse(GENERATED_SCENE)
root = tree.getroot()
compiler = root.find('compiler')
compiler.set('balanceinertia', 'true')
compiler.set('boundmass', '0.000001')
compiler.set('boundinertia', '0.000001')

option = root.find('option')
if option is None:
    option = ET.Element('option')
    root.insert(1, option)
option.set('gravity', '0 0 -9.81')
option.set('integrator', 'implicitfast')

asset = root.find('asset')
visual = root.find('visual')
if visual is None:
    visual = ET.Element('visual')
    root.insert(list(root).index(asset), visual)
ET.SubElement(visual, 'headlight', diffuse='0.7 0.7 0.7', ambient='0.35 0.35 0.35', specular='0.1 0.1 0.1')
ET.SubElement(visual, 'rgba', haze='0.15 0.25 0.35 1')
ET.SubElement(visual, 'global', azimuth='120', elevation='-20')
ET.SubElement(asset, 'texture', name='scene_skybox', type='skybox', builtin='gradient', rgb1='0.3 0.5 0.7', rgb2='0 0 0', width='512', height='3072')

worldbody = root.find('worldbody')
ET.SubElement(worldbody, 'light', name='ceiling_light', pos='0 0 6', dir='0 0 -1', directional='true', diffuse='0.8 0.8 0.8', specular='0.2 0.2 0.2')
ET.SubElement(worldbody, 'light', name='fill_light', pos='2 -2 4', dir='-0.3 0.3 -1', directional='true', diffuse='0.45 0.45 0.45', specular='0.1 0.1 0.1', castshadow='false')

ET.indent(tree, space='  ')
tree.write(GENERATED_SCENE, encoding='utf-8', xml_declaration=True)
model = mujoco.MjModel.from_xml_path(str(GENERATED_SCENE))
print(f'Compiled: {model.nbody} bodies, {model.njnt} joints, {model.nu} actuators, {model.ngeom} geoms, {model.nlight} lights')

## Launch the viewer

This starts MuJoCo with gravity and contacts enabled. It does not execute robot actions.

In [ ]:
simulator = MujocoSimulator(
    file_path=str(GENERATED_SCENE),
    _headless=HEADLESS,
    _step_size=0.001,
    config={
        'gravity': True,
        'contact': True,
        'integrator': mujoco.mjtIntegrator.mjINT_IMPLICITFAST,
    },
)
simulator.set_joints_values(initial_joint_targets)

# Position actuators must use the same targets or reset/physics pulls joints back.
joint_to_actuator = {
    simulator._mj_model.joint(simulator._mj_model.actuator_trnid[index, 0]).name:
        simulator._mj_model.actuator(index).name
    for index in range(simulator._mj_model.nu)
}
for joint_name, target in initial_joint_targets.items():
    actuator_name = joint_to_actuator.get(joint_name)
    if actuator_name is not None:
        simulator.get_actuator(actuator_name).result.ctrl[0] = target

simulator.save(key_name='home')
synchronizer = MujocoSynchronizer(_world=world, simulator=simulator)
simulator.start(simulate_in_thread=False)
print('PR2 apartment scene started with parked arms and closed drawers.')

## Simple PyCRAM action plan

The stock Giskard-backed motions currently fail to compile for this parsed MJCF world. These notebook-local `ActionDescription` classes keep the PyCRAM sequential-plan structure while commanding the same SDT joints and MuJoCo position actuators deterministically.

In [ ]:
def interpolate_joint_targets(action, targets, duration):
    sim = action.simulator
    model = sim._mj_model
    data = sim._mj_data
    starts = {}
    qpos_addresses = {}
    actuator_ids = {}
    for joint_name in targets:
        joint_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, joint_name)
        qpos_addresses[joint_name] = int(model.jnt_qposadr[joint_id])
        starts[joint_name] = float(data.qpos[qpos_addresses[joint_name]])
        actuator_name = joint_to_actuator.get(joint_name)
        if actuator_name is not None:
            actuator_ids[joint_name] = mujoco.mj_name2id(
                model, mujoco.mjtObj.mjOBJ_ACTUATOR, actuator_name
            )

    steps = max(1, int(duration * 30))
    for step in range(1, steps + 1):
        t = step / steps
        alpha = t * t * (3.0 - 2.0 * t)
        for joint_name, target in targets.items():
            value = starts[joint_name] + alpha * (target - starts[joint_name])
            data.qpos[qpos_addresses[joint_name]] = value
            if joint_name in actuator_ids:
                data.ctrl[actuator_ids[joint_name]] = value
        mujoco.mj_step(model, data)
        sim.renderer.sync()
        if not sim.headless:
            time.sleep(duration / steps)

    for joint_name, target in targets.items():
        data.qpos[qpos_addresses[joint_name]] = target
        if joint_name in actuator_ids:
            data.ctrl[actuator_ids[joint_name]] = target
        action.world.get_connection_by_name(joint_name).position = target
    data.qvel[:] = 0.0
    mujoco.mj_forward(model, data)
    action.world.notify_state_change()


@dataclass
class ParkPR2ArmsAction(ActionDescription):
    simulator: MujocoSimulator
    duration: float = 2.0

    def execute(self):
        targets = {}
        for arm in self.context.robot.arms:
            parked = arm.get_joint_state_by_type(StaticJointState.PARK)
            targets.update(
                {connection.name.name: value for connection, value in parked.items()}
            )
        interpolate_joint_targets(self, targets, self.duration)


@dataclass
class RaisePR2TorsoAction(ActionDescription):
    simulator: MujocoSimulator
    duration: float = 1.5

    def execute(self):
        high = self.context.robot.torso.get_joint_state_by_type(TorsoState.HIGH)
        targets = {connection.name.name: value for connection, value in high.items()}
        interpolate_joint_targets(self, targets, self.duration)


@dataclass
class NavigatePR2Action(ActionDescription):
    simulator: MujocoSimulator
    x: float
    y: float
    yaw: float
    duration: float = 3.0

    def execute(self):
        targets = {
            'base_x_joint': self.x,
            'base_y_joint': self.y,
            'base_yaw_joint': self.yaw,
        }
        # Preserve the current targets for torso, arms, head, and grippers.
        model = self.simulator._mj_model
        data = self.simulator._mj_data
        for actuator_index in range(model.nu):
            joint_id = int(model.actuator_trnid[actuator_index, 0])
            joint_name = model.joint(joint_id).name
            targets.setdefault(joint_name, float(data.ctrl[actuator_index]))
        interpolate_joint_targets(self, targets, self.duration)

In [ ]:
# With yaw=-pi/2, reducing y moves the robot forward through the clear aisle.
NAVIGATION_GOAL = (1.2, 3.2, -1.5707963267948966)
simple_plan = sequential(
    [
        ParkPR2ArmsAction(simulator),
        RaisePR2TorsoAction(simulator),
        NavigatePR2Action(simulator, *NAVIGATION_GOAL),
    ],
    context=context,
).plan
print('Plan: park arms -> raise torso -> move forward -> stop')

## Execute the plan

Run this cell when you are ready to see the simple action sequence. The final position actuator targets hold the robot stopped at the navigation goal.

In [ ]:
simple_plan.perform()
print(
    'Plan finished:',
    f"torso={world.get_connection_by_name('torso_lift_joint').position:.3f}",
    f"base=({world.get_connection_by_name('base_x_joint').position:.3f}, "
    f"{world.get_connection_by_name('base_y_joint').position:.3f})",
)

## Stop before rerunning

In [ ]:
if 'synchronizer' in globals():
    synchronizer.stop()
if 'simulator' in globals():
    simulator.stop()
print('Scene stopped.')